# Data Layer — Libraries in Berlin
---
### 🧪 Step 1: Research & Data Modelling

### 🎯 Objective
The goal of Step 1 was to identify and document reliable data sources for libraries in Berlin. This step focused on discovering, collecting, and evaluating datasets suitable for integration into the Berlin data foundation.

---
### 🧩 Tasks Completed
- Created branch libraries-data-modelling and directory structure:
/libraries/
└── sources/

- Retrieved base data from OpenStreetMap (Overpass API) with amenity=library
- Loaded data into a GeoDataFrame using GeoPandas
- Inspected dataset structure and identified relevant fields
- Extracted the following key columns for modelling:
'library_id', 'name', 'name_en', 'alt_name', 'short_name', 'amenity', 'operator_type', 'type', 'operator', 'street', 'housenumber', 'postcode', 'city', 'country', 'latitude', 'longitude', 'geom_point', 'phone', 'contact_phone', 'email', 'contact_email', 'website', 'contact_website', 'opening_hours', 'wheelchair_accessible', 'toilets_wheelchair', 'level', 'access', 'network', 'internet_access', 'room_group_study', 'room_study_cabin', 'service_copy', 'service_scanner', 'isil_code', 'toilets'

- Exported raw data in both .csv and .geojson format into /sources
Created a detailed README.md inside /sources documenting:
Data source origin and update frequency
Relevant data fields
Transformation plan for Step 2
-----

### ✅ Outcome

A validated and documented dataset from **OpenStreetMap** with 148 entries representing various libraries across Berlin.
These data now serve as the foundation for Step 2: **Transformation & Preprocessing**.

---


In [129]:
#! pip install osmnx geopandas pandas --quiet
#! pip install geopy

In [130]:
# Imports

import osmnx as ox                  # fetch data from OpenStreetMap
import geopandas as gpd             # handle geospatial (map) data
import requests, pandas as pd       # work with tabular (spreadsheet) data
import os, json, time, random, re   # manage files, caching, and random processes
import numpy as np                  # numerical calculations, arrays
from geopy.geocoders import Nominatim, Photon       # convert addresses/places to coordinates
from geopy.extra.rate_limiter import RateLimiter    # slow down geocoding to avoid blocking
from tqdm.auto import tqdm                          # progress bar for loops
from shapely.geometry import Point                  # make point geometries for maps


1.1 Data Source Discovery

Conduct initial research to identify relevant data sources for libraries.

In [131]:
# Fetch Libraries in Berlin from OpenStreetMap(OSM) using the tag and place

# Define the place and the tag for libraries
place = "Berlin, Germany"
tags = {"amenity": "library"}

# Fetch the data as a GeoDataFrame
library_gdf = ox.features_from_place(place, tags)

# The 'library_gdf' GeoDataFrame now contains all OSM entities (nodes, ways, relations)
# within Berlin tagged as a library, along with their associated attributes.

library_gdf.describe(include='all').T

,count,unique,top,freq
geometry,149,149,POINT (13.3475136 52.5312448),1
addr:city,121,1,Berlin,121
addr:country,102,1,DE,102
addr:housenumber,127,81,1,10
addr:postcode,123,78,10117,11
...,...,...,...,...
name:tr,1,1,Berlin Eyalet Kütüphanesi,1
name:zh,1,1,柏林国立图书馆,1
wheelchair:url,1,1,https://staatsbibliothek-berlin.de/vor-ort/men...,1
year_of_construction,2,2,1903..1914,1


### 1.2 Modelling & Planning
Select and define the key attributes relevant to this layer.

**Example fields:**

- Library name
- Library type (public, university, research, district, cultural)
- Address
- District
- Neighborhood
- Contact info (phone, email, website)
- Opening hours
- Services (e.g., Wi-Fi, study rooms, events, digital catalog access)
- Accessibility features (e.g., wheelchair access, quiet study areas)
- Geolocation (lat/lon)
- Managing organization or parent institution

In [132]:
library_gdf = library_gdf.to_crs(epsg=4326)  # WGS84

In [133]:
gdf = library_gdf.copy()

# 1) Make sure the CRS is WGS84 (EPSG:4326)
if gdf.crs is None:
    gdf = gdf.set_crs(epsg=4326)
else:
    gdf = gdf.to_crs(epsg=4326)

# 2) Checks on geometry types
print(gdf.geom_type.value_counts(dropna=False).head())

# 3) Extract representative point for each geometry 
def to_point(geom):
    if geom is None or geom.is_empty:
        return None
    if geom.geom_type == "Point":
        return geom
    # For other geometry types, get a representative point
    try:
        return geom.representative_point()
    except Exception:
        # Fallback to centroid if representative_point fails
        return geom.centroid

gdf["geom_point"] = gdf.geometry.apply(to_point)
gdf = gdf[~gdf["geom_point"].isna()].copy()

# 4) Extracts Longitude/Latitude (x = lon, y = lat)
gdf["longitude"] = gdf["geom_point"].x
gdf["latitude"]  = gdf["geom_point"].y

# Final check of selected columns
gdf.head()

Point      108
Polygon     41
Name: count, dtype: int64


geometry addr:city addr:country  \
element id                                                            
node    29071031   POINT (13.34751 52.53124)    Berlin           DE   
        60848456   POINT (13.47084 52.53078)    Berlin           DE   
        203557001  POINT (13.53872 52.52816)    Berlin           DE   
        256922190   POINT (13.28719 52.5375)    Berlin           DE   
        257708789  POINT (13.20139 52.53613)    Berlin           DE   

                  addr:housenumber addr:postcode          addr:street  \
element id                                                              
node    29071031                33         10559   Perleberger Straße   
        60848456                14         10369  Anton-Saefkow-Platz   
        203557001                4         12681  Helene-Weigel-Platz   
        256922190               18         13627             Halemweg   
        257708789               13         13597   Carl-Schurz-Straße   

                           addr:suburb  amenity  check_date  \
element id                                                    
node    29071031                Moabit  library  2025-07-10   
        60848456                   NaN  library         NaN   
        203557001              Marzahn  library         NaN   
        256922190  Charlottenburg-Nord  library         NaN   
        257708789              Spandau  library         NaN   

                  check_date:opening_hours  ... name:no name:pl name:tr  \
element id                                  ...                           
node    29071031                2025-07-10  ...     NaN     NaN     NaN   
        60848456                2025-03-04  ...     NaN     NaN     NaN   
        203557001                      NaN  ...     NaN     NaN     NaN   
        256922190                      NaN  ...     NaN     NaN     NaN   
        257708789                      NaN  ...     NaN     NaN     NaN   

                  name:zh wheelchair:url year_of_construction roof:material  \
element id                                                                    
node    29071031      NaN            NaN                  NaN           NaN   
        60848456      NaN            NaN                  NaN           NaN   
        203557001     NaN            NaN                  NaN           NaN   
        256922190     NaN            NaN                  NaN           NaN   
        257708789     NaN            NaN                  NaN           NaN   

                                  geom_point  longitude   latitude  
element id                                                          
node    29071031   POINT (13.34751 52.53124)  13.347514  52.531245  
        60848456   POINT (13.47084 52.53078)  13.470838  52.530777  
        203557001  POINT (13.53872 52.52816)  13.538715  52.528158  
        256922190   POINT (13.28719 52.5375)  13.287186  52.537504  
        257708789  POINT (13.20139 52.53613)  13.201386  52.536133  

[5 rows x 154 columns]

In [134]:
# Reset index to get the OSM IDs as a column
gdf = gdf.reset_index()

gdf.head(2)

,element,id,geometry,addr:city,addr:country,addr:housenumber,addr:postcode,addr:street,addr:suburb,amenity,...,name:no,name:pl,name:tr,name:zh,wheelchair:url,year_of_construction,roof:material,geom_point,longitude,latitude
0,node,29071031,POINT (13.34751 52.53124),Berlin,DE,33,10559,Perleberger Straße,Moabit,library,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (13.34751 52.53124),13.347514,52.531245
1,node,60848456,POINT (13.47084 52.53078),Berlin,DE,14,10369,Anton-Saefkow-Platz,NaN,library,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (13.47084 52.53078),13.470838,52.530777


In [135]:
# List the columns names of the GeoData

gdf.columns.to_list()

['element',
 'id',
 'geometry',
 'addr:city',
 'addr:country',
 'addr:housenumber',
 'addr:postcode',
 'addr:street',
 'addr:suburb',
 'amenity',
 'check_date',
 'check_date:opening_hours',
 'description',
 'name',
 'opening_hours',
 'phone',
 'ref:isil',
 'toilets:wheelchair',
 'website',
 'wheelchair',
 'wheelchair:description',
 'wikidata',
 'wikimedia_commons',
 'wikipedia',
 'contact:phone',
 'internet_access',
 'internet_access:fee',
 'internet_access:operator',
 'contact:email',
 'contact:website',
 'garden:type',
 'note',
 'operator',
 'wheelchair:description:de',
 'email',
 'name:en',
 'operator:type',
 'operator:wikidata',
 'ref',
 'access',
 'changing_table',
 'kids_area:indoor',
 'level',
 'source',
 'toilets',
 'toilets:access',
 'alt_name',
 'contact:fax',
 'outdoor_seating',
 'fax',
 'room:group_study',
 'room:study_cabin',
 'service:copy',
 'service:scanner',
 'addr:housename',
 'addr:place',
 'layer',
 'name:etymology:wikidata',
 'air_conditioning',
 'name:de',
 'opera

🔑 Key Attributes for a Library Layer
| Key Attribute         | Source Field(s)                                                                 | Definition                                                                                                                   |
|-----------------------|----------------------------------------------------------------------------------|-------------------------------------------------------------------------------------------------------------------------------|
| Library Name          | name, name:en, alt_name, short_name                                             | The primary, official name of the library. Use name or the most appropriate language version if available.                    |
| Library Type          | amenity, operator:type, type, operator                                          | The function or category (e.g., public library – implied by `amenity=library`, university library, research library) and the type of managing body. |
| Address               | addr:street, addr:housenumber, addr:postcode, addr:city, addr:country           | The full physical street address for navigation and location.                                                                 |
| Geolocation           | latitude, longitude, geom_point                                                  | The geographic coordinates (latitude and longitude) of the library's location.                                                |
| Contact Info          | phone, contact:phone, email, contact:email, website, contact:website            | Key methods for contacting the library (phone number, email address, and official website URL).                               |
| Opening Hours         | opening_hours                                                                    | The regular schedule indicating when the library is open to the public.                                                       |
| Accessibility         | wheelchair, toilets:wheelchair, level, access                                   | Indicators of physical accessibility, primarily for mobility (e.g., wheelchair access status).                                 |
| Managing Organization | operator, operator:type, network                                                | The name or type of the organization, institution, or network that runs the library.                                          |
| Core Services         | internet_access, room:group_study, room:study_cabin, service:copy, service:scanner, toilets | Availability of essential on-site resources like Wi-Fi (internet_access), study rooms, and other key services.                |


 - **Creating New Table with Selected Columns**

In [136]:

key_attributes = [
    'id','name', 'name:en', 'alt_name', 'short_name', 'amenity', 'operator:type', 
    'type', 'operator', 'addr:street', 'addr:housenumber', 'addr:postcode', 
    'addr:city', 'addr:country', 'latitude', 'longitude', 'geom_point', 
    'phone', 'contact:phone', 'email', 'contact:email', 'website', 
    'contact:website', 'opening_hours', 'wheelchair', 'toilets:wheelchair', 
    'level', 'access', 'network', 'internet_access', 'room:group_study', 
    'room:study_cabin', 'service:copy', 'service:scanner', 'ref:isil', 'toilets'
]

gdf = gdf[key_attributes]


# Rename map for only the columns that need renaming

rename_map = {
    "id": "library_id",
    "name:en": "name_en",
    "addr:street": "street",
    "addr:housenumber": "housenumber",
    "addr:postcode": "postcode",
    "addr:city": "city",
    "addr:country": "country",
    "opening_hours": "opening_hours",
    "operator:type": "operator_type",

    # Alternative contact channels
    "contact:phone": "contact_phone",
    "contact:email": "contact_email",
    "contact:website": "contact_website",

    "wheelchair": "wheelchair_accessible",
    "toilets:wheelchair": "toilets_wheelchair",
    "service:copy": "service_copy",
    "service:scanner": "service_scanner",
    "room:group_study": "room_group_study",
    "room:study_cabin": "room_study_cabin",
    "ref:isil": "isil_code"
}


gdf =  gdf.rename(columns=rename_map)

# Print the columns to confirm renaming worked
print("Columns after renaming:", gdf.columns.tolist())


Columns after renaming: ['library_id', 'name', 'name_en', 'alt_name', 'short_name', 'amenity', 'operator_type', 'type', 'operator', 'street', 'housenumber', 'postcode', 'city', 'country', 'latitude', 'longitude', 'geom_point', 'phone', 'contact_phone', 'email', 'contact_email', 'website', 'contact_website', 'opening_hours', 'wheelchair_accessible', 'toilets_wheelchair', 'level', 'access', 'network', 'internet_access', 'room_group_study', 'room_study_cabin', 'service_copy', 'service_scanner', 'isil_code', 'toilets']


- **Checking Missing Values**


In [137]:
# Count total number of missing values per column
missing_count = gdf.isna().sum().sort_values(ascending=False)

# Build table with counts and % of missing values
# This is important to understand data quality - so we can drop the columns with too many missing values

missing = pd.DataFrame({
    "missing_count": missing_count,
    "missing_pct": (missing_count / len(gdf) * 100).round(1)
}).sort_values(by="missing_pct", ascending=False)

display(missing)

,missing_count,missing_pct
service_scanner,148,99.3
room_study_cabin,148,99.3
room_group_study,148,99.3
network,148,99.3
type,144,96.6
service_copy,142,95.3
access,140,94.0
toilets,140,94.0
short_name,139,93.3
alt_name,139,93.3


- **Distinct Values per Column**

In [138]:
# Number of unique values per column
# Goal: See the “variety” of each column.

distinct = gdf.nunique().sort_values(ascending=False)
print(distinct)

library_id               149
longitude                149
geom_point               149
latitude                 149
name                     142
street                   115
opening_hours            108
housenumber               81
postcode                  78
website                   63
isil_code                 52
contact_website           48
contact_phone             43
phone                     40
operator                  32
email                     29
contact_email             19
name_en                   14
level                     11
alt_name                  10
short_name                10
operator_type              4
internet_access            4
wheelchair_accessible      3
access                     3
toilets_wheelchair         3
type                       1
amenity                    1
country                    1
city                       1
room_group_study           1
network                    1
room_study_cabin           1
service_copy               1
service_scanne

- **Most Common Values per Column**

In [139]:
# Example: most common wheelchair accessible
print("\nHow many institutions have wheelchair accessibility:")
print(gdf["wheelchair_accessible"].value_counts().head(10))


How many institutions have wheelchair accessibility:
wheelchair_accessible
yes        74
limited    19
no         10
Name: count, dtype: int64


In [140]:
print("\nTop 10 streets:")
print(gdf["street"].value_counts().head(10))


Top 10 streets:
street
Straße des 17. Juni    3
Frankfurter Allee      2
Potsdamer Straße       2
Greifswalder Straße    2
Malteserstraße         2
Hauptstraße            2
Bat-Yam-Platz          2
Unter den Linden       2
Dorotheenstraße        2
Garystraße             2
Name: count, dtype: int64


In [141]:
# Example: most common postcode 
print("\nTop 10 postcodes:")
print(gdf["postcode"].value_counts().head(10))


Top 10 postcodes:
postcode
10117    11
14195    10
10623     5
10785     4
10115     3
10999     3
13353     3
12353     2
14059     2
10965     2
Name: count, dtype: int64


In [142]:
# Example: most common opening_hours
print("\nTop 10 opening hours:")
print(gdf["opening_hours"].value_counts().head(10))


Top 10 opening hours:
opening_hours
Mo-Fr 09:00-20:00                                                                                                                                        4
Mo-Fr 10:00-19:30; Sa 10:00-14:00                                                                                                                        3
Mo-Sa 08:00-22:00                                                                                                                                        2
Mo-Fr 09:00-17:00                                                                                                                                        2
Mo-Sa 08:00-22:00; Su 10:00-18:00                                                                                                                        2
24/7                                                                                                                                                     2
Mo,Th 13:00-19:00; Tu,We,Fr 13:00

In [143]:
# Goal: Verify lat/lon look realistic.
# Why? If values are way off, something went wrong in conversion.

print("Latitude range:", gdf["latitude"].min(), "to", gdf["latitude"].max())

print("Longitude range:", gdf["longitude"].min(), "to", gdf["longitude"].max())

Latitude range: 52.3859756 to 52.6356441
Longitude range: 13.1429895 to 13.6214013


In [144]:
gdf.shape

(149, 36)

#### 1.3 Prepare the /sources Directory
- Raw Data Files:

  * libraries_raw.geojson (includes geometry)
  * libraries_raw.csv (tabular only, no geometry)

- README.md in /sources will contain:

  * Data sources used.
  * Planned transformation steps.

In [ ]:
# os.chdir(os.path.join(os.path.expanduser("~"), "s"))
# os.getcwd()

In [ ]:
# Define file paths
raw_geojson_path = "sources/geojson/library_raw.geojson"
raw_csv_path = "sources/csv/library_raw.csv"


# Save as GeoJSON (keeps geometry) and CSV
# Save csv & geojson to correct folders in sources

gdf.to_csv(raw_csv_path, index=False)
gdf.to_file(raw_geojson_path, driver="GeoJSON")


print(f"Raw data saved to: {raw_geojson_path} and {raw_csv_path}")

#### 1.4 Review
- All draft 36 target columns defined.
- Data sources identified and documented.
- Schema draft created.
- Data fetched and stored in /sources.
- Data cleaning & enrichment plan in place.

**Next Step:** Step 2 — Fetch & Transform data.

## 🧭 Step 2: Data Transformation & Preprocessing
🎯 Objective

The goal of Step 2 is to write transformation scripts in Python or SQL, to enrich, clean and normalize raw data through standardize (naming conventions, address formats, and category labels), Validate geolocation coordinates, Merge and deduplicate entries across multiple sources to produce a unified version ready for database integration.

---

**Address Enrichment  — Reverse Geocoding with Nominatim**

- Objective: Fill missing address details (street, housenumber, postcode, city, country) using latitude and longitude.
- Filtering: Processed only records missing key address fields to save API calls.
- API:
  * Nominatim (OpenStreetMap) as the geocoding service.
- Rate limiting: Used RateLimiter with a 1.5-second delay between requests to respect API rules.
- Field extraction: Filled in and standardized street, housenumber, postcode, city, and country.
- Error handling: Skipped records when geocoding failed, leaving them as missing.
- Progress: Progress bar showed enrichment status.
- Result: Most missing fields were enriched and the dataset is now more complete.

In [145]:
# Use geopy(Geopy Nominatim) with rate limiting to fill in missing address components (street, housenumber, postcode) for rows that have coordinates but lack address details.
# Crucial Step: You need to instantiate the geocoding service and apply rate limiting.

#from geopy.geocoders import Nominatim
#from geopy.extra.rate_limiter import RateLimiter


# ---1 Setup Nominatim Geocoder ---
# Use a custom user agent for politeness as required by Nominatim
geolocator = Nominatim(user_agent="libraries_in_berlin")
geocode = RateLimiter(geolocator.reverse, min_delay_seconds=1.5) # Limit to 1 call every 1.5 seconds

# --- 2 Define Enrichment Function ---
def fill_missing_address(row):
    # Only try to geocode if essential address components are missing
    if pd.isna(row['street']) or pd.isna(row['postcode']):
        try:
            # Reverse geocode using the existing lat/lon
            location = geocode((row['latitude'], row['longitude']), exactly_one=True, language='en')
            
            if location and location.raw and 'address' in location.raw:
                address = location.raw['address']
                
                # Update missing street
                if pd.isna(row['street']):
                    row['street'] = address.get('road')
                
                # Update missing house number
                if pd.isna(row['housenumber']):
                    row['housenumber'] = address.get('house_number')
                
                # Update missing postcode
                if pd.isna(row['postcode']):
                    row['postcode'] = address.get('postcode')

                # Update missing city
                if pd.isna(row['city']):
                    row['city'] = address.get('city')

                # Update missing country
                if pd.isna(row['country']):
                    row['country'] = address.get('country')
                    
        except Exception as e:
            # print(f"Geocoding error for ID {row['library_id']}: {e}")
            pass # Skip and leave NaNs

    return row

# --- 3 Apply Enrichment (WARNING: This can be slow!) ---
print("\nStarting Nominatim Address Enrichment (slow due to rate limiting)...")
# Filter to only the rows that are missing address data and then apply
rows_to_enrich = gdf[(gdf['street'].isna()) | (gdf['housenumber'].isna()) | (gdf['postcode'].isna()) | (gdf['city'].isna()) | (gdf['country'].isna())].index

if not rows_to_enrich.empty:
    tqdm.pandas(desc="Geocoding Missing Addresses")
    gdf.loc[rows_to_enrich] = gdf.loc[rows_to_enrich].progress_apply(fill_missing_address, axis=1)
else:
    print("No rows require Nominatim enrichment.")

print("Nominatim Enrichment Complete.")


Starting Nominatim Address Enrichment (slow due to rate limiting)...


Geocoding Missing Addresses: 100%|██████████| 49/49 [00:37<00:00,  1.30it/s]

Nominatim Enrichment Complete.


**Data Cleaning and quality improvement**

- Sparse columns removed: Dropped columns with >80% missing data unless potentially useful for future work.
- Constant value fields imputed: Filled missing ‘city’ and ‘country’ entries with 'Berlin' and 'DE' since all libraries are in Berlin, Germany.
- Contact info consolidated: Combined email and phone fields into unified columns, then dropped redundant fields for clarity.
- Duplicates removed: Deleted duplicate libraries based on unique library_id to ensure each record is unique.
- Column standardization: Renamed columns to snake_case and replaced special characters for consistency.
- Type consistency: Set appropriate data types, such as category for accessibility and string for opening hours.
- Geometry validation: Excluded rows with invalid or missing geometry to keep only valid points.

In [146]:

# 2.1 Remove columns with > 80% missing data (unless useful for metadata).

cols_to_drop = [
    "service_scanner", "room_study_cabin", "room_group_study",
    "network", "type", "service_copy", "toilets", "access",
    "short_name", "alt_name", "name_en"
]

gdf = gdf.drop(columns=cols_to_drop, errors="ignore") # Drop sparse columns


In [147]:
# 2.2 Impute Constant Columns ---or --- Drop irrelevant/redundant columns (e.g., city, country)

# Since all libraries are in Berlin, Germany(DE), these should not be missing and are not dropped.
gdf['city'] = gdf['city'].fillna('Berlin')
gdf['country'] = gdf['country'].fillna('DE')


# gdf = gdf.drop(columns=['city', 'country'], errors='ignore') # Drop the original columns (or keep, depending on preference)


# 2.3 Consolidate Contact Channels
print("Consolidating website, phone and email fields...")
gdf['final_email'] = gdf['contact_email'].combine_first(gdf['email'])
gdf['final_phone'] = gdf['contact_phone'].combine_first(gdf['phone'])
gdf["website_url"] = gdf["contact_website"].fillna(gdf["website"])

# Drop the old, redundant contact fields
gdf = gdf.drop(columns=['contact_email', 'email', 'contact_phone', 'phone', 'contact_website', 'website'], errors='ignore')


Consolidating website, phone and email fields...


In [148]:
# 2.3 Remove duplicates

gdf = gdf.drop_duplicates(subset=['library_id'], keep='first') # Remove duplicates based on the primary ID
'''
# Use name + coordinates:
# gdf = gdf.drop_duplicates(subset=["name", "latitude", "longitude"])
'''
#---

# 2.4 Standardize column names and ensure consistent formats.
'''Snake-case + no special characters:'''

gdf.columns = (
    gdf.columns
    .str.lower()
    .str.replace(":", "_")
    .str.replace("-", "_")
)

#----

# 2.5 Ensure correct dtypes

gdf["opening_hours"] = gdf["opening_hours"].astype("string")
gdf["wheelchair_accessible"] = gdf["wheelchair_accessible"].astype("category")


#----

# 2.6 Remove invalid geometries (Validate geometry)

gdf = gdf[gdf.geom_point.notnull()]

print(f"\nDataFrame size after cleaning: {len(gdf)} rows, {len(gdf.columns)} columns")


DataFrame size after cleaning: 149 rows, 22 columns


In [149]:
# 4. FINAL CHECK
# ==============================================================================

print("\n--- Final Cleaned DataFrame Info ---")
print(f"Total Rows: {len(gdf)}")
print(f"Total Columns: {len(gdf.columns)}")
print("\nFinal Column List:")
print(gdf.columns.tolist())
print("\nFinal DataFrame Info:")
gdf.info()
print("\nNew Missing Value Table (Top 10):")
print(gdf.isna().sum().sort_values(ascending=False).head(10))


--- Final Cleaned DataFrame Info ---
Total Rows: 149
Total Columns: 22

Final Column List:
['library_id', 'name', 'amenity', 'operator_type', 'operator', 'street', 'housenumber', 'postcode', 'city', 'country', 'latitude', 'longitude', 'geom_point', 'opening_hours', 'wheelchair_accessible', 'toilets_wheelchair', 'level', 'internet_access', 'isil_code', 'final_email', 'final_phone', 'website_url']

Final DataFrame Info:
<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 149 entries, 0 to 148
Data columns (total 22 columns):
 #   Column                 Non-Null Count  Dtype   
---  ------                 --------------  -----   
 0   library_id             149 non-null    int64   
 1   name                   147 non-null    object  
 2   amenity                149 non-null    object  
 3   operator_type          38 non-null     object  
 4   operator               54 non-null     object  
 5   street                 148 non-null    object  
 6   housenumber            132 non-null

### **🗺️ Spatial Join/Integration: Assigning District and Neighbourhood**

To enrich each libraries with its administrative context;

- **Load districts**: Imported official Berlin boundaries from lor_ortsteile.geojson.
- **Perform spatial join**: The spatial join (using predicate="within") matches each institution’s geometry with the polygon representing the *Ortsteil*(neighbourhood) in which it lies.
- **Rename columns**: Columns BEZIRK (district) and OTEIL (neighbourhood) were renamed to district and neighbourhood for consistency.
- **Add district IDs**: Mapped each district to its **official statistical ID**(district_id) according to Regionalstatistik Berlin/Brandenburg.
- These standardized district identifiers ensure that all institutions can be integrated with other city datasets (e.g., population, demographics, infrastructure).

This enrichment step closes the remaining geographic gaps left after geocoding, ensuring every record is now spatially anchored within a known Berlin district and neighbourhood.

In [150]:
# Load official Berlin districts GeoDataFrame from lor_ortsteile.geojson

berlin_districts_gdf = gpd.read_file("sources/lor_ortsteile.geojson")
print(berlin_districts_gdf.columns)

Index(['gml_id', 'spatial_name', 'spatial_alias', 'spatial_type', 'OTEIL',
       'BEZIRK', 'FLAECHE_HA', 'geometry'],
      dtype='object')


In [151]:
berlin_districts_gdf.head(2)

,gml_id,spatial_name,spatial_alias,spatial_type,OTEIL,BEZIRK,FLAECHE_HA,geometry
0,re_ortsteil.0101,0101,Mitte,Polygon,Mitte,Mitte,1063.8748,"POLYGON ((13.41649 52.52696, 13.41635 52.52702..."
1,re_ortsteil.0102,0102,Moabit,Polygon,Moabit,Mitte,768.7909,"POLYGON ((13.33884 52.51974, 13.33884 52.51974..."


In [152]:
# --- 1 . Reproject GeoDataFrames to EPSG:4326 ---

# Set the geometry column for gdf and ensure the CRS is 4326
gdf = gdf.set_geometry("geom_point").to_crs(epsg=4326) 

# Ensure the district data is also 4326 for a valid spatial join
berlin_districts_gdf = berlin_districts_gdf.to_crs(epsg=4326)

In [ ]:
# sets "geom_point" as the“active” geometry column to use for mapping and spatial joins.
# gdf = gdf.set_geometry("geom_point")

In [ ]:
# --- 2. Perform the Spatial Join ---

# We are using 'spatial_name' as the unique ID (district_id) and selecting the names
library_df_district = gpd.sjoin(
    gdf,
    berlin_districts_gdf[["BEZIRK", "OTEIL", "spatial_name", "geometry"]],
    how="left",
    predicate="within"
)
# --- 3. Rename Columns to Match SQL Schema ---

library_df_district = library_df_district.rename(columns={
    "BEZIRK": "district",       # Matches SQL 'district' (Bezirk name)
    "OTEIL": "neighbourhood",     # Matches SQL 'neighbourhood' (Ortsteil name)
    "spatial_name": "neighbourhood_id" # Matches SQL 'district_id' (Foreign Key)
}).drop(columns=["index_right"])  # Drop the index column added by sjoin

In [156]:
# Generating district ids


# District mapping (official codes as strings)
district_mapping = {
    'Mitte': '11001001',
    'Friedrichshain-Kreuzberg': '11002002',
    'Pankow': '11003003',
    'Charlottenburg-Wilmersdorf': '11004004',
    'Spandau': '11005005',
    'Steglitz-Zehlendorf': '11006006',
    'Tempelhof-Schöneberg': '11007007',
   'Neukölln': '11008008',
    'Treptow-Köpenick': '11009009',
    'Marzahn-Hellersdorf': '11010010',
    'Lichtenberg': '11011011',
    'Reinickendorf': '11012012'
}

# Apply mapping to create district_id column (string)
library_df_district['district_id'] = library_df_district['district'].map(district_mapping).astype(str)

print("\nSpatial join complete. Ready for WKT conversion and database upload.")
display(library_df_district[['district_id', 'district', 'neighbourhood', 'neighbourhood_id']].head())


Spatial join complete. Ready for WKT conversion and database upload.


,district_id,district,neighbourhood,neighbourhood_id
0,11001001,Mitte,Moabit,0102
1,11011011,Lichtenberg,Fennpfuhl,1111
2,11010010,Marzahn-Hellersdorf,Marzahn,1001
3,11004004,Charlottenburg-Wilmersdorf,Charlottenburg-Nord,0406
4,11005005,Spandau,Spandau,0501


In [157]:
print("✅ Dataset after cleaning and transforming\n")

# Shape of dataframe
print(f"Number of rows: {library_df_district.shape[0]}")
print(f"Number of columns: {library_df_district.shape[1]}")

# Column list
print("\nRemaining columns:")
print(library_df_district.columns.tolist())

# Missing values check
missing = library_df_district.isnull().sum()
print("\nMissing values after cleaning and transforming :")
print(missing)

✅ Dataset after cleaning and transforming

Number of rows: 149
Number of columns: 26

Remaining columns:
['library_id', 'name', 'amenity', 'operator_type', 'operator', 'street', 'housenumber', 'postcode', 'city', 'country', 'latitude', 'longitude', 'geom_point', 'opening_hours', 'wheelchair_accessible', 'toilets_wheelchair', 'level', 'internet_access', 'isil_code', 'final_email', 'final_phone', 'website_url', 'district', 'neighbourhood', 'neighbourhood_id', 'district_id']

Missing values after cleaning and transforming :
library_id                 0
name                       2
amenity                    0
operator_type            111
operator                  95
street                     1
housenumber               17
postcode                   0
city                       0
country                    0
latitude                   0
longitude                  0
geom_point                 0
opening_hours             31
wheelchair_accessible     46
toilets_wheelchair       113
level    

**Export & Documentation**

- Export cleaned and unified dataset:
  * library_unified_data.csv
- Include clear Markdown documentation:
- Summary of enrichment and cleaning steps
- Data quality notes

 
 

#### ✅ Step 2 — Final Summary Documentation
This notebook successfully implements the **transformation and preprocessing pipeline** for the *Libraries in Berlin* data layer.


| Column Name            | Description                                                                                       | Example Value |
|-------------------------|---------------------------------------------------------------------------------------------------|----------------|
| **library_id**          | Unique identifier of the library (from OSM or assigned).                                         | 6235274853 |
| **name**                | Official or commonly used library name.                                                          | Zentral- und Landesbibliothek Berlin |
| **amenity**             | OSM tag indicating the feature type (always “library”).                                          | library |
| **operator_type**       | Type of managing organization (e.g., public, university, private).                               | public |
| **operator**            | Organization or network responsible for managing the library.                                    | VÖBB – Verbund der Öffentlichen Bibliotheken Berlins |
| **street**              | Street name from address fields.                                                                 | Breite Straße |
| **housenumber**         | House or building number of the address.                                                         | 36 |
| **postcode**            | Postal code of the library’s location.                                                           | 10178 |
| **city**                | City name.                                                                                       | Berlin |
| **country**             | Country name or ISO code.                                                                        | DE |
| **latitude**            | Geographic latitude coordinate (EPSG:4326).                                                      | 52.520832 |
| **longitude**           | Geographic longitude coordinate (EPSG:4326).                                                     | 13.409528 |
| **geom_point**          | Simplified geometry used for spatial joins (Point, EPSG:4326).                                   | POINT (13.409528 52.520832) |
| **geometry**            | Original geometry object from OSM (Polygon or MultiPolygon).                                     | POLYGON ((13.405 52.52, 13.406 52.52, ...)) |
| **opening_hours**       | Library’s official opening hours (OSM or external enrichment).                                   | Mo–Fr 09:00–19:00; Sa 10:00–14:00 |
| **wheelchair_accessible** | Accessibility status for wheelchair users.                                                      | yes |
| **toilets_wheelchair**  | Indicates if toilets are accessible to wheelchair users.                                         | yes |
| **level**               | Level or floor number within a building (if specified).                                          | 1 |
| **internet_access**     | Indicates if public internet or Wi-Fi is available.                                              | wlan |
| **isil_code**           | International Standard Identifier for Libraries (if available).                                  | DE-B1511 |
| **final_email**         | Cleaned and consolidated email contact from all sources.                                         | info@voebb.de |
| **final_phone**         | Cleaned and consolidated phone number from all sources.                                          | +49 30 90226 401 |
| **website_url**         | Verified or merged website URL (from OSM, VÖBB, or Wikidata).                                   | https://www.zlb.de |
| **district**            | Administrative district (Bezirk) within Berlin, from spatial join.                              | Mitte |
| **neighbourhood**       | Smaller local area or neighbourhood (Ortsteil).                                                 | Tiergarten |
| **district_id**         | Unique ID or code for the district (from Berlin Open Data).                                      | 01 |


---
**Key Steps**
- **Address Enrichment**: Filled missing address fields using Nominatim reverse geocoding, with caching and rate limiting.
- **Data Cleaning**: Removed columns with over 80% missing values,  dropped duplicates and invalid geometries.
- **Normalization**: Standardized variants and formats.
- **Spatial Integration**: Added district and neighbourhood using official lor_ortsteile.geojson and generated district_id.
- **Validation**: Checked coordinate ranges and ensured all geometries were valid.
- **Final Output:** Saved unified dataset library_unified_data.csv with standardized schema for MVP integration.

**Notes on Data Quality**
- **Completeness**: Core fields (like name and location) are strong; some optional tags remain sparse.
- **Consistency**: Standardized across all categories and coordinate systems.
- **Reliability**: Good for spatial aggregation and dashboard-level analytics; limited for detailed attribute completeness

---

**Next Step**: Load dataset into the database (Step 3: Integration).

# 🚀 Step 3: Database Integration: Schema & Constraints

## Workflow

### 1. Database Schema Definition (CREATE TABLE)

To properly integrate the new layer, we must define the table structure using an explicit `CREATE TABLE` statement, rather than just loading the data.

**Action:** Execute the `CREATE TABLE` statement to define our `libraries_layer` table.

**Mandatory Constraints:** Define the following constraints to enforce data quality and establish relationships:

- **PRIMARY KEY:** Uniquely identifies each record of the libraries layer (e.g., `library_id`, etc).
- **NOT NULL:** Ensure essential fields like `district_id` and `name` always contain a value.
- **FOREIGN KEY:** Establish the crucial link to the existing `berlin_data.districts` layer.

---


### 2. Validation & Quality Checks (Python/Pandas Phase)

Perform data validation and integrity checks prior to database insertion to ensure schema compatibility, data consistency and data quality.

These checks include:
- Verifying that required columns are not null (`library_id`, `name`, `district_id`).
- Ensuring geometries are valid (`gdf.geometry.is_valid`).
- Detecting duplicates or orphaned references (`district_id` linkage).
- Confirming coordinate ranges fall within Berlin’s bounding box.


In [158]:
# --- 1. Configuration (Define Constraints) ---

# Fields required to be NOT NULL in the SQL table
NOT_NULL_FIELDS = [
    'library_id', 'name', 'amenity', 'postcode', 'city', 'country',
    'latitude', 'longitude', 'district_id', 'district'
]

# Coordinate bounds for Berlin (Data Accuracy Check)
BERLIN_BOUNDS = {
    'lat_min': 52.33, 'lat_max': 52.68,
    'lon_min': 13.08, 'lon_max': 13.77
}

# SQL VARCHAR length limits to check against
MAX_LENGTHS = {
    'name': 255,
    'library_id': 20, # VARCHAR(20) check
    'district_id': 8  # Full 8-digit ID (e.g., '11001001')
}

# --- 2. Validation Functions ---

def run_completeness_check(df: pd.DataFrame, fields: list) -> bool:
    """Checks for missing values in mandatory NOT NULL fields."""
    print("### A. Completeness Check (NOT NULL) ###")
    
    # Calculate missing values for the mandatory fields
    missing_data = df[fields].isnull().sum()
    critical_missing = missing_data[missing_data > 0]

    if critical_missing.empty:
        print("✅ PASS: All mandatory fields are 100% complete.")
        return True
    else:
        print("❌ FAIL: The following mandatory fields have missing values:")
        print(critical_missing)
        # Display sample rows with issues for quick debugging
        null_rows = df[df[critical_missing.index].isnull().any(axis=1)]
        print(f"\nSample of {len(null_rows)} problematic rows:")
        print(null_rows[['library_id', 'name'] + critical_missing.index.tolist()].head())
        return False


def run_consistency_check(df: pd.DataFrame, lengths: dict) -> bool:
    """Checks data types and ensures field content respects SQL length constraints."""
    print("\n### B. Consistency Check (Types & Lengths) ###")
    passed = True
    
    # 1. Primary Key Type Conversion (library_id)
    # Convert to pandas nullable integer (Int64) first, then to string for SQL VARCHAR match
    try:
        # Fill NA temporarily with a marker, convert, then replace marker with None/NaN if needed later
        df['library_id'] = df['library_id'].astype('Int64').astype(str).replace('<NA>', np.nan)
        print("✅ PASS: library_id successfully standardized to string type.")
    except Exception as e:
        print(f"❌ FAIL: library_id conversion error: {e}")
        passed = False
    
    # 2. Length Constraint Check (e.g., name, library_id)
    for col, max_len in lengths.items():
        # Only check strings and non-missing values
        long_items = df[df[col].astype(str).str.len() > max_len]
        if not long_items.empty:
            print(f"❌ FAIL: {col} has {len(long_items)} item(s) exceeding VARCHAR({max_len}).")
            passed = False
        else:
            print(f"✅ PASS: {col} length checked against VARCHAR({max_len}) limit.")
            
    return passed


def run_accuracy_check(df: pd.DataFrame, bounds: dict) -> bool:
    """Validates geographic coordinates fall within the expected Berlin boundaries."""
    print("\n### C. Data Accuracy Check (Geographic Bounds) ###")

    # Check bounds simultaneously for latitude and longitude
    out_of_bounds = df[
        (df['latitude'] < bounds['lat_min']) | (df['latitude'] > bounds['lat_max']) |
        (df['longitude'] < bounds['lon_min']) | (df['longitude'] > bounds['lon_max'])
    ]

    if out_of_bounds.empty:
        print("✅ PASS: All coordinates fall within the expected Berlin bounding box.")
        return True
    else:
        print(f"❌ FAIL: {len(out_of_bounds)} record(s) are outside Berlin's bounds.")
        print(out_of_bounds[['library_id', 'latitude', 'longitude']].head())
        return False


def run_relational_check(df: pd.DataFrame) -> bool:
    """Checks district_id format and lists unique IDs for external parent table verification."""
    print("\n### D. Relational Integrity Check (District ID) ###")

    # Check if district_id is exactly 8 digits (to match the full ASGS key)
    # Ensure it's treated as a string before checking the length/pattern
    id_series = df['district_id'].astype(str).str.strip()
    
    # Check if all non-missing IDs match the 8-digit pattern
    valid_format = id_series.dropna().str.match(r'^\d{8}$').all()
    
    if valid_format:
        print("✅ PASS: All district_id values match the required 8-digit format.")
    else:
        # Identify the rows that fail the 8-digit check
        invalid_format_count = (~id_series.dropna().str.match(r'^\d{8}$')).sum()
        print(f"❌ FAIL: {invalid_format_count} district_id values have an incorrect format (expected 8 digits).")

    # List unique IDs found in the data (Crucial step for FOREIGN KEY check)
    unique_ids = sorted(id_series.dropna().unique().tolist())
    print(f"\nUnique district_id values found: {unique_ids}")
    print("ACTION: You must manually ensure these full 8-digit IDs exist in the 'berlin_data.districts' table.")
    
    return valid_format


# --- 3. Execution Block ---

# ----------------------------------------------------------------------------------
# load the data transformed in Step 2:

# FILE_PATH = './libraries/unified/library_unified_data.csv' 
# library_gdf = pd.read_csv('library_unified_data.csv') 
# library_gdf
# ----------------------------------------------------------------------------------

print(f"\n--- Starting Data Validation for {len(library_df_district)} Records from {'library_unified_data.csv'} ---")


# Run all checks and collect results
is_complete = run_completeness_check(library_df_district.copy(), NOT_NULL_FIELDS)
is_consistent = run_consistency_check(library_df_district.copy(), MAX_LENGTHS)
is_accurate = run_accuracy_check(library_df_district.copy(), BERLIN_BOUNDS)
is_relational_ok = run_relational_check(library_df_district.copy())

# Final Summary
print("\n--- FINAL VALIDATION SUMMARY ---")
if all([is_complete, is_consistent, is_accurate, is_relational_ok]):
    print("🎉 **SUCCESS!** The dataset is ready for final database insertion.")
else:
    print("⚠️ **FAILURE!** Review the '❌ FAIL' messages above and clean the data before loading.")


--- Starting Data Validation for 149 Records from library_unified_data.csv ---
### A. Completeness Check (NOT NULL) ###
❌ FAIL: The following mandatory fields have missing values:
name    2
dtype: int64

Sample of 2 problematic rows:
      library_id name name
71    5062614778  NaN  NaN
102  11929111653  NaN  NaN

### B. Consistency Check (Types & Lengths) ###
✅ PASS: library_id successfully standardized to string type.
✅ PASS: name length checked against VARCHAR(255) limit.
✅ PASS: library_id length checked against VARCHAR(20) limit.
✅ PASS: district_id length checked against VARCHAR(8) limit.

### C. Data Accuracy Check (Geographic Bounds) ###
✅ PASS: All coordinates fall within the expected Berlin bounding box.

### D. Relational Integrity Check (District ID) ###
✅ PASS: All district_id values match the required 8-digit format.

Unique district_id values found: ['11001001', '11002002', '11003003', '11004004', '11005005', '11006006', '11007007', '11008008', '11009009', '11010010', '

In [159]:
# Check A. Completeness Fix: Missing Names
print("--- Fixing Missing Names ---")

# Identify the rows with missing names
name_nan_indices = library_df_district[library_df_district['name'].isnull()].index

if not name_nan_indices.empty:
    for idx in name_nan_indices:
        # Create a placeholder name using the library_id and district
        placeholder_name = (
            f"Unnamed Library - ID {library_df_district.loc[idx, 'library_id']} in District {library_df_district.loc[idx, 'district']}"
        )
        library_df_district.loc[idx, 'name'] = placeholder_name
        print(f"Fixed row {idx}: Name set to '{placeholder_name}'")

    # Re-run the completeness check for the 'name' column
    if library_df_district['name'].isnull().sum() == 0:
        print("✅ SUCCESS: All 'name' fields are now populated.")
    else:
        print("❌ ERROR: Name fixation failed.")
else:
    print("No missing names found.")

--- Fixing Missing Names ---
Fixed row 71: Name set to 'Unnamed Library - ID 5062614778 in District Mitte'
Fixed row 102: Name set to 'Unnamed Library - ID 11929111653 in District Mitte'
✅ SUCCESS: All 'name' fields are now populated.


In [160]:
# --- Final Export of Fixed Data ---
OUTPUT_PATH = 'sources/library_unified_data.csv'
library_df_district.to_csv(OUTPUT_PATH, index=False)
print(f"\n✅ SUCCESS: Fixed dataset exported to {OUTPUT_PATH}")




✅ SUCCESS: Fixed dataset exported to sources/library_unified_data.csv


**Populate Database**

CREATE TABLE IF NOT EXISTS test_berlin_data.libraries (
    library_id BIGINT PRIMARY KEY,
    
    -- Basic details, often requiring text fields (VARCHAR).
	name VARCHAR(255) NOT NULL,
	amenity VARCHAR(50) NOT NULL, -- Mandatory field for the type of amenity (e.g., 'library')
	operator_type VARCHAR(100),
	operator VARCHAR(500),
	
	-- Address fields. 
    street VARCHAR(150),
    housenumber VARCHAR(10),
    postcode VARCHAR(10) NOT NULL,
    city VARCHAR(50) NOT NULL,
    country VARCHAR(50) NOT NULL,
    
    -- Geographic coordinates. NUMERIC(9, 6) is suitable for high precision (up to 6 decimal places).
    latitude NUMERIC(9, 6) NOT NULL,
    longitude NUMERIC(9, 6) NOT NULL,
    
    -- District information, used for linking to the 'districts' table.
    district_id VARCHAR(50) NOT NULL,
    district VARCHAR(100) NOT NULL,
    neighbourhood VARCHAR(100),
    neighbourhood_id VARCHAR(100),
    
    -- Contact and service details.
    final_email VARCHAR(255),
    final_phone VARCHAR(100),
    website_url VARCHAR(255),
    opening_hours VARCHAR(255),
    isil_code VARCHAR(100),
    
    -- Accessibility and internet info.
    wheelchair_accessible VARCHAR(10),
    toilets_wheelchair VARCHAR(10),
    internet_access VARCHAR(100),
    "level" VARCHAR(10),  -- 'level' is a reserved keyword in some SQL dialects, so it's best practice to quote it (e.g., "level")
    
    -- Spatial Data: GEOMETRY(Point, 4326) is excellent for storing the location point.
    geom_point GEOMETRY(Point, 4326),
    
    -- Foreign Key Constraint: Links this table to the 'districts' table.
    -- Ensures that every 'district_id' in 'libraries' exists in the 'districts' table. 
    CONSTRAINT district_id_fk
        FOREIGN KEY (district_id)
        REFERENCES  test_berlin_data.districts(district_id)
        ON DELETE RESTRICT -- Prevents deleting a district if libraries still reference it
        ON UPDATE CASCADE  -- Automatically updates the district_id in this table if it changes in the 'districts' table

);



In [161]:
import psycopg2
import pandas as pd
from sqlalchemy import create_engine, text
import warnings

warnings.filterwarnings("ignore")

In [ ]:
user_name=''
password=''

In [ ]:
# Conection
host = 'localhost'
port = '5433'
database = ''
schema=''

#connection to db after you opened tunnel
engine = create_engine(f'postgresql+psycopg2://{user_name}:{password}@{host}:{port}/{database}')

In [164]:
# Drop the Old Table (Important!)

from sqlalchemy import text

drop_table_query = f"DROP TABLE IF EXISTS {schema}.libraries CASCADE;"
with engine.connect() as conn:
    conn.execute(text(drop_table_query))
    conn.commit()
print("Old 'libraries' table dropped successfully.")

Old 'libraries' table dropped successfully.


In [165]:
#this is where you create table with constraints and references first
# SQL query to create the libraries table with constraints and foreign key

create_table_query = f"""
CREATE TABLE IF NOT EXISTS {schema}.libraries (
    -- 🔑 PRIMARY KEY (Mandatory)
    library_id BIGINT PRIMARY KEY,             -- Unique library identifier (from OSM, large integer)

    -- 🏛️ Administrative & Naming Fields
    name VARCHAR(255) NOT NULL,                -- Official name of the library
    amenity VARCHAR(50) NOT NULL,              -- OSM tag: always 'library'
    operator_type VARCHAR(100),                 -- Type of managing organization (public, university, etc.)
    operator VARCHAR(500),                     -- Name of the managing organization (e.g., VÖBB)

    -- 📍 Address Fields
    street VARCHAR(150),                       -- Street name
    housenumber VARCHAR(10),                   -- Building number
    postcode VARCHAR(10) NOT NULL,             -- Postal code
    city VARCHAR(50) NOT NULL,                 -- City name (e.g., Berlin)
    country VARCHAR(50) NOT NULL,               -- Country code (e.g., DE)

    -- 🌐 Geographic Coordinates
    latitude NUMERIC(9, 6) NOT NULL,           -- Latitude (EPSG:4326)
    longitude NUMERIC(9, 6) NOT NULL,          -- Longitude (EPSG:4326)

    -- 🗺️ Administrative Context (Link to Districts Table)
    district_id VARCHAR(50) NOT NULL,           -- 2-digit district code (FK to berlin_data.districts)
    district VARCHAR(100) NOT NULL,             -- District (Bezirk) name
    neighbourhood VARCHAR(100),                 -- Local area (Ortsteil)
    neighbourhood_id VARCHAR(100),              -- Local area ID (Foreign Key)

    -- ☎️ Contact & Service Fields
    final_email VARCHAR(255),                  -- Consolidated email
    final_phone VARCHAR(100),                   -- Consolidated phone number
    website_url VARCHAR(255),                  -- Official or verified website
    opening_hours VARCHAR(255),                -- Opening hours (OSM or enriched)
    isil_code VARCHAR(100),                     -- International Standard Identifier for Libraries

    -- ♿ Accessibility & Facilities
    wheelchair_accessible VARCHAR(10),         -- Accessibility info (yes/no/limited)
    toilets_wheelchair VARCHAR(10),            -- Wheelchair-accessible toilets (yes/no)
    internet_access VARCHAR(100),               -- Internet/Wi-Fi availability
    "level" VARCHAR(10),                       -- Floor/level number (quoted: reserved SQL keyword)

    
    -- 📍 PostGIS Geometry Column
    geom_point GEOMETRY(Point, 4326),          -- Spatial point geometry (WGS84 coordinates)


    -- 🔗 FOREIGN KEY Constraint (REFERENCES the 8-digit ID in the parent table)
    CONSTRAINT district_id_fk
        FOREIGN KEY (district_id)
        REFERENCES berlin_source_data.districts(district_id)
        ON DELETE RESTRICT                     -- Prevent deletion of parent district if libraries exist
        ON UPDATE CASCADE                      -- Update district_id in libraries if it changes upstream
);

"""

# Execute the query to create empty table
with engine.connect() as conn:
    conn.execute(text(create_table_query))
    conn.commit()  # commit the transaction

In [166]:
# 1. Run the WKT Conversion - Convert the geometry column from Shapely objects to WKT strings.
# Ensure you run this only once on your GeoDataFrame after it has been loaded and before the final step.

library_df_district['geom_point'] = library_df_district['geom_point'].apply(
    lambda geom: geom.wkt if geom is not None else None
)


In [167]:
# Insert the Data
#  Send the DataFrame to the database using .to_sql()
library_df_district.to_sql(
    'libraries',      
    engine,
    schema=schema,
    if_exists='append', # ✅ keeps table, just inserts data
    index=False
)


print("DataFrame (with WKT geometryand district IDs) successfully loaded into PostgreSQL!")

DataFrame (with WKT geometryand district IDs) successfully loaded into PostgreSQL!


In [169]:
##let's query test data!
query = f"""
SELECT * 
from berlin_source_data.libraries
"""

# Execute the query
with engine.connect() as conn:
    df= pd.read_sql(text(query), conn)
    conn.commit()  # commit the transaction
df.head(2)

,library_id,name,amenity,operator_type,operator,street,housenumber,postcode,city,country,...,final_email,final_phone,website_url,opening_hours,isil_code,wheelchair_accessible,toilets_wheelchair,internet_access,level,geom_point
0,29071031,Bruno-Lösche-Bibliothek,library,None,None,Perleberger Straße,33,10559,Berlin,DE,...,None,+49 30901833025,http://www.berlin.de/stadtbibliothek-mitte/bib...,Mo-Fr 10:00-19:30; Sa 10:00-14:00,DE-B707,yes,no,None,None,0101000020E610000098D4754DEDB12A40C51561D4FF43...
1,60848456,Anton-Saefkow-Bibliothek,library,None,None,Anton-Saefkow-Platz,14,10369,Berlin,DE,...,None,+4930902963773,http://www.berlin.de/ba-lichtenberg/auf-einen-...,"Sa 09:00-18:00; Mo,Tu 09:00-19:00; We 13:00-19...",None,yes,None,None,None,0101000020E61000003E6A5DB411F12A4032BBDD81F043...
